# Etapa 5: Entrenamiento, Tuning y Evaluación de la Solución Predictiva

Este cuaderno realiza la separación del conjunto de datos de entrenamiento/validación de manera aislada por clientes, el tuning de hiperparámetros del estimador RandomForest mediante RandomizedSearchCV acoplado en un preprocesamiento StandardScaler/OneHotEncoder, y calcula las métricas de negocio Accuracy@1 y Accuracy@3 de manera vectorizada en NumPy.

In [1]:
import sys
import os
from pathlib import Path

# Resolver la ruta raíz del proyecto de forma dinámica y portable
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "pandas", "scikit-learn", "joblib", "numpy"])
    print("Dependencias instaladas en Colab.")
    
    # Intentar montar Google Drive automáticamente si no está montado
    if not Path('/content/drive').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            print("No se pudo montar Drive automáticamente. Por favor, móntelo en el panel izquierdo de Colab.")

current_dir = Path(os.getcwd()).resolve()
if IN_COLAB:
    # Rutas de búsqueda comunes en Google Drive y Colab
    possible_paths = [
        Path('/content/drive/MyDrive/Colab Notebooks/Proyecto'),
        Path('/content/drive/MyDrive/Proyecto'),
        Path('/content/Proyecto'),
        Path('/content')
    ]
    for p in possible_paths:
        if (p / "notebooks").exists():
            current_dir = p / "notebooks"
            break

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

## 1. Lanzar el Entrenamiento y Evaluación
Ejecutamos el script de modelado predictivo, que guardará el modelo serializado en `models/modelo_next_best_product.pkl` y las métricas en `models/metrics.json`.

In [2]:
from src.train_model import train_model

train_model(PROJECT_ROOT)

2026-06-15 20:05:23,503 - INFO - [ML] Cargando matrices analíticas de la capa Gold...


2026-06-15 20:05:23,592 - INFO - [ML] Entrenamiento: 11955 filas | Validación: 2992 filas


2026-06-15 20:05:23,594 - INFO - [ML] Iniciando optimización de hiperparámetros (Tuning Layer)...


2026-06-15 20:05:32,052 - INFO - [ML] Mejores parámetros: {'classifier__n_estimators': 100, 'classifier__min_samples_leaf': 2, 'classifier__max_depth': 14}


2026-06-15 20:05:32,100 - INFO - [ML] Validación ROC-AUC: 77.7374%


...

2026-06-15 20:05:32,104 - INFO - [ML] Validación Accuracy@3: 23.0000%


2026-06-15 20:05:32,105 - INFO - [ML] Calculando importancia de las variables para explicabilidad...


2026-06-15 20:05:32,143 - INFO - [ML] Top 5 variables más influyentes: {'renta': 0.44527, 'categoria_producto_candidato_Cuenta': 0.19591, 'edad': 0.15279, 'categoria_producto_candidato_Depósito': 0.03736, 'categoria_producto_candidato_Inversión': 0.028}


2026-06-15 20:05:32,145 - INFO - [ML] Métricas y explicabilidad guardadas en: /Volumes/HD/0.2.Sistemas_de_Informacion_UG/11vo-semestre/ANÁLISIS DE DATOS MASIVO/Proyecto/models/metrics.json


2026-06-15 20:05:32,214 - INFO - [ML] Pipeline de inferencia guardado en: /Volumes/HD/0.2.Sistemas_de_Informacion_UG/11vo-semestre/ANÁLISIS DE DATOS MASIVO/Proyecto/models/modelo_next_best_product.pkl


## 2. Inspección de Métricas Guardadas

In [3]:
import json

metrics_path = PROJECT_ROOT / "models" / "metrics.json"
with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("=== RENDIMIENTO DEL MODELO EN VALIDACIÓN ===")
print(f"ROC-AUC: {metrics['roc_auc']:.4%}")
print(f"Accuracy@1 (Top 1 Recomendación): {metrics['accuracy_at_1']:.4%}")
print(f"Accuracy@3 (Top 3 Recomendaciones): {metrics['accuracy_at_3']:.4%}")
print("Mejores Hiperparámetros:", metrics["best_params"])

=== RENDIMIENTO DEL MODELO EN VALIDACIÓN ===
ROC-AUC: 77.7374%
Accuracy@1 (Top 1 Recomendación): 19.0000%
Accuracy@3 (Top 3 Recomendaciones): 23.0000%
Mejores Hiperparámetros: {'classifier__n_estimators': 100, 'classifier__min_samples_leaf': 2, 'classifier__max_depth': 14}
